# Differential Gene Expression Analysis in Fatal COVID-19

## Introduction

Gene expression analysis provides a way to investigate how biological activity differs between experimental or clinical conditions.

In this project, RNA sequencing data from human tissue samples associated with fatal COVID-19 will be analysed to investigate differences in gene expression between COVID-19 cases and healthy controls.

The dataset is derived from the E-ENAD-46 study, which profiled lung and colon tissue from individuals who died from severe COVID-19 alongside healthy control samples.

The analysis will focus on the gene-expression data and will use Python to perform data inspection, preprocessing, exploratory analysis and statistical testing.

Because RNA-seq data are high-dimensional, thousands of genes may be tested simultaneously. The analysis will therefore consider both the magnitude of expression differences and the problem of multiple statistical comparisons.

This project is intended as an educational bioinformatics analysis rather than a clinical or diagnostic study.

## Objectives

The objectives of this analysis are to:

1. Inspect and understand the structure of the RNA-seq expression dataset.
2. Examine the available sample metadata and identify relevant comparison groups.
3. Preprocess the gene-expression measurements for exploratory analysis.
4. Compare gene-expression patterns between COVID-19 and control samples.
5. Quantify differences in gene expression using fold changes and statistical testing.
6. Apply multiple-testing correction when assessing gene-level statistical significance.
7. Visualise major patterns in the transcriptomic data.
8. Interpret the findings in a biological context while considering the limitations of the dataset and analytical approach.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 20)

In [2]:
import os

os.listdir("data")

['.DS_Store',
 'E-ENAD-46.g3_g1.go.gsea.tsv',
 'E-ENAD-46.g4_g2.reactome.gsea.tsv',
 'E-ENAD-46.condensed-sdrf.tsv',
 'E-ENAD-46.idf.txt',
 'E-ENAD-46-raw-counts.tsv']

In [3]:
counts_path = "data/E-ENAD-46-raw-counts.tsv"

counts = pd.read_csv(
    counts_path,
    sep="\t"
)

counts.head()

,Gene ID,Gene Name,SRR12816718,SRR12816719,SRR12816720,SRR12816721,SRR12816722,SRR12816723,SRR12816724,SRR12816725,SRR12816726,SRR12816727,SRR12816728,SRR12816729,SRR12816730,SRR12816731,SRR12816732,SRR12816733,SRR12816734,SRR12816735,SRR12816736,SRR12951211,SRR12951212,SRR12951213,SRR12951214,SRR12951215,SRR12951216,SRR12951217,SRR12951218,SRR12951219,SRR12951220,SRR12951221,SRR12951222,SRR12951223,SRR12951224,SRR12951225,SRR12951226,SRR12951227,SRR12951228,SRR12951229
0,ENSG00000000003,TSPAN6,245,369,331,380,704,138,713,1051,574,537,234,169,624,289,2980,475,444,14,169,512,1608,201,84,99,695,10,71,303,1559,934,1044,1196,391,1130,550,326,568,464
1,ENSG00000000005,TNMD,6,28,45,1,0,3,4,2,9,35,66,14,48,21,0,15,2,0,0,5,0,6,0,2,1,0,0,11,1,26,8,19,5,13,18,2,23,8
2,ENSG00000000419,DPM1,195,260,133,332,647,321,822,708,298,125,427,219,351,207,526,257,344,14,619,315,66,593,343,151,458,39,377,364,489,369,288,405,311,368,281,252,271,168
3,ENSG00000000457,SCYL3,216,245,38,109,385,163,207,394,312,384,517,289,568,279,344,366,322,209,325,270,179,428,194,155,822,79,277,267,495,351,387,534,235,595,491,396,365,325
4,ENSG00000000460,C1orf112,234,131,82,87,179,62,181,182,357,119,760,154,220,198,909,224,237,44,317,240,49,207,51,86,235,32,64,136,187,210,226,302,204,404,349,376,261,215


In [4]:
counts.shape

(58735, 40)

In [5]:
counts.columns.tolist()

['Gene ID',
 'Gene Name',
 'SRR12816718',
 'SRR12816719',
 'SRR12816720',
 'SRR12816721',
 'SRR12816722',
 'SRR12816723',
 'SRR12816724',
 'SRR12816725',
 'SRR12816726',
 'SRR12816727',
 'SRR12816728',
 'SRR12816729',
 'SRR12816730',
 'SRR12816731',
 'SRR12816732',
 'SRR12816733',
 'SRR12816734',
 'SRR12816735',
 'SRR12816736',
 'SRR12951211',
 'SRR12951212',
 'SRR12951213',
 'SRR12951214',
 'SRR12951215',
 'SRR12951216',
 'SRR12951217',
 'SRR12951218',
 'SRR12951219',
 'SRR12951220',
 'SRR12951221',
 'SRR12951222',
 'SRR12951223',
 'SRR12951224',
 'SRR12951225',
 'SRR12951226',
 'SRR12951227',
 'SRR12951228',
 'SRR12951229']

In [7]:
with open("data/E-ENAD-46.condensed-sdrf.tsv", "r", encoding="utf-8") as f:
    for i in range(10):
        print(repr(f.readline()))

'E-ENAD-46\t\tSRR12816718\tcharacteristic\tage\tnot available\n'
'E-ENAD-46\t\tSRR12816718\tcharacteristic\tbiosource provider\tGuoping Wang Lab, Department of Pathology, School of Basic Medicine, Tongji Medical College, Huazhong University of Science and Technology\n'
'E-ENAD-46\t\tSRR12816718\tcharacteristic\tdisease\tnormal\thttp://purl.obolibrary.org/obo/PATO_0000461\n'
'E-ENAD-46\t\tSRR12816718\tcharacteristic\tindividual\tWuhan_control_1\n'
'E-ENAD-46\t\tSRR12816718\tcharacteristic\torganism\tHomo sapiens\thttp://purl.obolibrary.org/obo/NCBITaxon_9606\n'
'E-ENAD-46\t\tSRR12816718\tcharacteristic\torganism part\tlung\thttp://purl.obolibrary.org/obo/UBERON_0002048\n'
'E-ENAD-46\t\tSRR12816718\tcharacteristic\tsex\tnot available\n'
'E-ENAD-46\t\tSRR12816718\tcharacteristic\tspecimen with known storage state\tparaffin specimen\thttp://purl.obolibrary.org/obo/OBI_0000950\n'
'E-ENAD-46\t\tSRR12816718\tfactor\tdisease\tnormal\thttp://purl.obolibrary.org/obo/PATO_0000461\n'
'E-ENAD-46\t\

In [8]:
metadata = pd.read_csv(
    "data/E-ENAD-46.condensed-sdrf.tsv",
    sep="\t",
    header=None,
    names=[
        "study",
        "sample",
        "run",
        "record_type",
        "attribute",
        "value",
        "ontology"
    ],
    engine="python"
)

metadata.head(10)

,study,sample,run,record_type,attribute,value,ontology
0,E-ENAD-46,NaN,SRR12816718,characteristic,age,not available,None
1,E-ENAD-46,NaN,SRR12816718,characteristic,biosource provider,"Guoping Wang Lab, Department of Pathology, Sch...",None
2,E-ENAD-46,NaN,SRR12816718,characteristic,disease,normal,http://purl.obolibrary.org/obo/PATO_0000461
3,E-ENAD-46,NaN,SRR12816718,characteristic,individual,Wuhan_control_1,None
4,E-ENAD-46,NaN,SRR12816718,characteristic,organism,Homo sapiens,http://purl.obolibrary.org/obo/NCBITaxon_9606
5,E-ENAD-46,NaN,SRR12816718,characteristic,organism part,lung,http://purl.obolibrary.org/obo/UBERON_0002048
6,E-ENAD-46,NaN,SRR12816718,characteristic,sex,not available,None
7,E-ENAD-46,NaN,SRR12816718,characteristic,specimen with known storage state,paraffin specimen,http://purl.obolibrary.org/obo/OBI_0000950
8,E-ENAD-46,NaN,SRR12816718,factor,disease,normal,http://purl.obolibrary.org/obo/PATO_0000461
9,E-ENAD-46,NaN,SRR12816718,factor,organism part,lung,http://purl.obolibrary.org/obo/UBERON_0002048


In [9]:
metadata.shape

(380, 7)

In [10]:
metadata.columns

Index(['study', 'sample', 'run', 'record_type', 'attribute', 'value',
       'ontology'],
      dtype='object')

In [11]:
metadata["attribute"].value_counts()

attribute
disease                              76
organism part                        76
age                                  38
biosource provider                   38
individual                           38
organism                             38
sex                                  38
specimen with known storage state    38
Name: count, dtype: int64

In [12]:
metadata.loc[
    metadata["attribute"] == "disease",
    ["sample", "run", "value"]
].head(20)

,sample,run,value
2,NaN,SRR12816718,normal
8,NaN,SRR12816718,normal
12,NaN,SRR12816719,COVID-19
18,NaN,SRR12816719,COVID-19
22,NaN,SRR12816720,COVID-19
28,NaN,SRR12816720,COVID-19
32,NaN,SRR12816721,COVID-19
38,NaN,SRR12816721,COVID-19
42,NaN,SRR12816722,COVID-19
48,NaN,SRR12816722,COVID-19


In [13]:
metadata.loc[
    metadata["attribute"] == "organism part",
    ["sample", "run", "value"]
].head(20)

,sample,run,value
5,NaN,SRR12816718,lung
9,NaN,SRR12816718,lung
15,NaN,SRR12816719,lung
19,NaN,SRR12816719,lung
25,NaN,SRR12816720,lung
29,NaN,SRR12816720,lung
35,NaN,SRR12816721,lung
39,NaN,SRR12816721,lung
45,NaN,SRR12816722,lung
49,NaN,SRR12816722,lung


In [14]:
disease_metadata = metadata[
    (metadata["record_type"] == "factor") &
    (metadata["attribute"] == "disease")
][["run", "value"]].copy()

disease_metadata = disease_metadata.rename(
    columns={"value": "disease"}
)

disease_metadata

,run,disease
8,SRR12816718,normal
18,SRR12816719,COVID-19
28,SRR12816720,COVID-19
38,SRR12816721,COVID-19
48,SRR12816722,COVID-19
...,...,...
338,SRR12951225,normal
348,SRR12951226,normal
358,SRR12951227,normal
368,SRR12951228,normal


In [15]:
disease_metadata.shape

(38, 2)

In [16]:
disease_metadata["disease"].value_counts()

disease
normal      20
COVID-19    18
Name: count, dtype: int64

In [17]:
disease_metadata["run"].nunique()

38

In [18]:
tissue_metadata = metadata[
    (metadata["record_type"] == "factor") &
    (metadata["attribute"] == "organism part")
][["run", "value"]].copy()

tissue_metadata = tissue_metadata.rename(
    columns={"value": "tissue"}
)

tissue_metadata.head()

,run,tissue
9,SRR12816718,lung
19,SRR12816719,lung
29,SRR12816720,lung
39,SRR12816721,lung
49,SRR12816722,lung


In [19]:
sample_metadata = disease_metadata.merge(
    tissue_metadata,
    on="run",
    how="inner"
)

sample_metadata.head()

,run,disease,tissue
0,SRR12816718,normal,lung
1,SRR12816719,COVID-19,lung
2,SRR12816720,COVID-19,lung
3,SRR12816721,COVID-19,lung
4,SRR12816722,COVID-19,lung


In [20]:
sample_metadata.shape

(38, 3)

In [21]:
sample_metadata["tissue"].value_counts()

tissue
lung     19
colon    19
Name: count, dtype: int64

In [22]:
sample_metadata["disease"].value_counts()

disease
normal      20
COVID-19    18
Name: count, dtype: int64

In [23]:
pd.crosstab(
    sample_metadata["tissue"],
    sample_metadata["disease"]
)

disease,COVID-19,normal
tissue,,
colon,9,10
lung,9,10


In [24]:
lung_metadata = sample_metadata[
    sample_metadata["tissue"] == "lung"
].copy()

lung_metadata

,run,disease,tissue
0,SRR12816718,normal,lung
1,SRR12816719,COVID-19,lung
2,SRR12816720,COVID-19,lung
3,SRR12816721,COVID-19,lung
4,SRR12816722,COVID-19,lung
5,SRR12816723,COVID-19,lung
6,SRR12816724,COVID-19,lung
7,SRR12816725,COVID-19,lung
8,SRR12816726,normal,lung
9,SRR12816727,normal,lung


In [25]:
lung_metadata["disease"].value_counts()

disease
normal      10
COVID-19     9
Name: count, dtype: int64

In [26]:
counts.shape

(58735, 40)

In [27]:
counts.head()

,Gene ID,Gene Name,SRR12816718,SRR12816719,SRR12816720,SRR12816721,SRR12816722,SRR12816723,SRR12816724,SRR12816725,SRR12816726,SRR12816727,SRR12816728,SRR12816729,SRR12816730,SRR12816731,SRR12816732,SRR12816733,SRR12816734,SRR12816735,SRR12816736,SRR12951211,SRR12951212,SRR12951213,SRR12951214,SRR12951215,SRR12951216,SRR12951217,SRR12951218,SRR12951219,SRR12951220,SRR12951221,SRR12951222,SRR12951223,SRR12951224,SRR12951225,SRR12951226,SRR12951227,SRR12951228,SRR12951229
0,ENSG00000000003,TSPAN6,245,369,331,380,704,138,713,1051,574,537,234,169,624,289,2980,475,444,14,169,512,1608,201,84,99,695,10,71,303,1559,934,1044,1196,391,1130,550,326,568,464
1,ENSG00000000005,TNMD,6,28,45,1,0,3,4,2,9,35,66,14,48,21,0,15,2,0,0,5,0,6,0,2,1,0,0,11,1,26,8,19,5,13,18,2,23,8
2,ENSG00000000419,DPM1,195,260,133,332,647,321,822,708,298,125,427,219,351,207,526,257,344,14,619,315,66,593,343,151,458,39,377,364,489,369,288,405,311,368,281,252,271,168
3,ENSG00000000457,SCYL3,216,245,38,109,385,163,207,394,312,384,517,289,568,279,344,366,322,209,325,270,179,428,194,155,822,79,277,267,495,351,387,534,235,595,491,396,365,325
4,ENSG00000000460,C1orf112,234,131,82,87,179,62,181,182,357,119,760,154,220,198,909,224,237,44,317,240,49,207,51,86,235,32,64,136,187,210,226,302,204,404,349,376,261,215


In [28]:
counts.columns.tolist()

['Gene ID',
 'Gene Name',
 'SRR12816718',
 'SRR12816719',
 'SRR12816720',
 'SRR12816721',
 'SRR12816722',
 'SRR12816723',
 'SRR12816724',
 'SRR12816725',
 'SRR12816726',
 'SRR12816727',
 'SRR12816728',
 'SRR12816729',
 'SRR12816730',
 'SRR12816731',
 'SRR12816732',
 'SRR12816733',
 'SRR12816734',
 'SRR12816735',
 'SRR12816736',
 'SRR12951211',
 'SRR12951212',
 'SRR12951213',
 'SRR12951214',
 'SRR12951215',
 'SRR12951216',
 'SRR12951217',
 'SRR12951218',
 'SRR12951219',
 'SRR12951220',
 'SRR12951221',
 'SRR12951222',
 'SRR12951223',
 'SRR12951224',
 'SRR12951225',
 'SRR12951226',
 'SRR12951227',
 'SRR12951228',
 'SRR12951229']

In [29]:
lung_samples = lung_metadata["run"].tolist()

lung_samples

['SRR12816718',
 'SRR12816719',
 'SRR12816720',
 'SRR12816721',
 'SRR12816722',
 'SRR12816723',
 'SRR12816724',
 'SRR12816725',
 'SRR12816726',
 'SRR12816727',
 'SRR12816728',
 'SRR12816729',
 'SRR12816730',
 'SRR12816731',
 'SRR12816732',
 'SRR12816733',
 'SRR12816734',
 'SRR12816735',
 'SRR12816736']

In [30]:
set(lung_samples).issubset(counts.columns)

True

In [31]:
lung_counts = counts[
    ["Gene ID", "Gene Name"] + lung_samples
].copy()

lung_counts.shape

(58735, 21)

In [32]:
lung_counts["Gene ID"].duplicated().sum()

np.int64(0)

In [33]:
lung_counts["Gene Name"].duplicated().sum()

np.int64(19403)

In [34]:
lung_counts[lung_samples].dtypes.value_counts()

int64    19
Name: count, dtype: int64

In [35]:
lung_counts[lung_samples].describe().T.head()

,count,mean,std,min,25%,50%,75%,max
SRR12816718,58735.0,295.608751,11329.623509,0.0,0.0,1.0,37.0,2351642.0
SRR12816719,58735.0,467.121239,34683.719021,0.0,0.0,0.0,22.0,7891723.0
SRR12816720,58735.0,405.627650,36375.728944,0.0,0.0,0.0,15.0,8304690.0
SRR12816721,58735.0,336.109168,16043.921878,0.0,0.0,0.0,24.0,3236968.0
SRR12816722,58735.0,331.889316,12382.037767,0.0,0.0,0.0,25.0,2188714.0


## Dataset Structure

The expression dataset contains 58,735 annotated genes measured across 38 sequencing runs.

Sample metadata identified 19 lung tissue samples and 19 colon tissue samples. Each tissue contains nine COVID-19 samples and ten normal control samples.

For this analysis, only the lung tissue samples will be used. This produces a balanced comparison of nine COVID-19 lung samples against ten normal lung samples.

The expression matrix contains two gene annotation columns, `Gene ID` and `Gene Name`, followed by the sequencing-run identifiers representing individual samples.

Restricting the analysis to a single tissue reduces biological heterogeneity and allows the primary comparison to focus specifically on differences associated with COVID-19 status within lung tissue.